# PolyWhisper — Whisper Baselines (Resumable, Colab T4)

Run in order. If Colab disconnects: re-run all cells — completed work is skipped via state on HF Hub.

**Runtime**: ~5 min/lang on T4 (small), ~15 min (medium), ~30 min (large).

In [ ]:
#@title 1. Install CUDA-enabled PyTorch + deps
# Force CUDA build (Colab's default pip torch is sometimes CPU-only)
!pip install -q --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
!pip install -q datasets transformers soundfile huggingface_hub

import torch
print('torch:', torch.__version__, '| cuda build:', torch.version.cuda, '| available:', torch.cuda.is_available())

In [ ]:
#@title 2. Verify GPU + HF login
import torch
assert torch.cuda.is_available(), 'CUDA NOT available — Runtime → Change runtime type → T4 GPU → Restart runtime'
print('GPU OK:', torch.cuda.get_device_name(0))

from huggingface_hub import notebook_login
notebook_login()  # paste your HF token (read+write)

In [ ]:
#@title 3. Download script + run baselines
from huggingface_hub import hf_hub_download
hf_hub_download('eulogik/polywhisper', 'baselines_resumable.py', local_dir='./')

#@markdown Which size? (or set run_all below for all three)
size = 'small' #@param ['small', 'medium', 'large']
langs = 'hi,ta,te,bn,mr' #@param {type:'string'}
run_all = False #@param {type:'boolean'}

if run_all:
    for s in ['small', 'medium', 'large']:
        print(f'\n{"="*60}\n# BASELINE: {s}\n{"="*60}')
        !python baselines_resumable.py --repo eulogik/polywhisper --size {s} --langs {langs} --device cuda
else:
    !python baselines_resumable.py --repo eulogik/polywhisper --size {size} --langs {langs} --device cuda

In [ ]:
#@title 4. Check progress / generate paper table
from huggingface_hub import hf_hub_download
import json

repo = 'eulogik/polywhisper'
state = json.load(open(hf_hub_download(repo, 'baselines_state.json', repo_type='model')))
experts = json.load(open(hf_hub_download(repo, 'fleurs_normalized_results.json', repo_type='model')))

print('Last update:', state['last_update'])
sizes = sorted(set(k.split('_')[0] for k in state['completed']))
print('Completed:', len(state['completed']), '| sizes done:', sizes)
for k, v in sorted(state['completed'].items()):
    print(f"  {k}: WER {v['wer']:.1f}% CER {v['cer']:.1f}% ({v['n']} samples)")

print('\n| Lang | Expert WER/CER |', *[f'{s} WER/CER |' for s in sizes], 'script-match |')
print('|---|---|', *['---|' for _ in sizes], '---|')
for lang in ['hi','ta','te','bn','mr']:
    sm = experts.get(lang,{}).get('pure',{}).get('script_matched',{})
    e = f"{sm.get('wer',0):.1f} / {sm.get('cer',0):.1f}"
    row = f'| {lang} | {e} | '
    for s in sizes:
        c = state['completed'].get(f'{s}_{lang}')
        row += f"{c['wer']:.1f} / {c['cer']:.1f} | " if c else '— | '
    row += f"{sm.get('rate',0):.1f}% |"
    print(row)